# Hands-on with socio4health: Housing Deficit Colombia

This notebook demonstrates how to use the socio4health extractor to reproduce DANE tables for years prior to 2018, using microdata from the Quality of Life Survey (ECV), in order to calculate Colombia’s housing deficit.

In [ ]:
!pip install "socio4health[scraping]"

## 1. Imports

In [2]:
from socio4health import Extractor
from functools import reduce
import pandas as pd
from pathlib import Path
import uuid

## 2. ECV Data Configuration

In [3]:
ecv_data = {
    2010: "https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata",
    2011: "https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata",
    2012: "https://microdatos.dane.gov.co/index.php/catalog/124/get-microdata",
    2013: "https://microdatos.dane.gov.co/index.php/catalog/213/get-microdata",
    2014: "https://microdatos.dane.gov.co/index.php/catalog/342/get-microdata",
    2015: "https://microdatos.dane.gov.co/index.php/catalog/419/get-microdata",
    2016: "https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata",
    2017: "https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata"
}

vivienda_cols = [
    "DIRECTORIO", "YEAR", "CLASE", "REGION", "P1_DEPARTAMENTO", "P1_MUNICIPIO",
    "P4000", "P4005", "P4015", "P8520S1", "P8520S3", "P8520S4", "P8520S5", "P70", "CANT_HOGARES_VIVIENDA"
]

hogar_cols = [
    "DIRECTORIO", "YEAR", "CLASE", "REGION", "P1_DEPARTAMENTO", "P1_MUNICIPIO",
    "P205", "CANT_PERSONAS_HOGAR", "P5010", "P8526", "P8530", "P8532", "P764"
]

dfs_vivienda_by_year = {}
dfs_hogar_by_year = {}

## 3. Data Extraction by Year (SOCIO4HEALTH)

In [5]:
for year, url in ecv_data.items():
        print(f"\n{year}: {url}")
        
        if year == 2010:
            key_words = [
                r"(?i)datos[\s_]+(?:de[\s_]+)?identificaci[oó]n",
                r"(?i)servicios[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)composici[oó]n[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)datos[\s_]+(?:de[\s_]+)?(?:la[\s_]+)?vivienda",
            ]
        elif year == 2011:
            key_words = [
                r"(?i)identificaci[oó]n.*vivienda",
                r"(?i)vivienda.*identificaci[oó]n",
                r"(?i)datos[\s_]+de[\s_]+identificaci[oó]n",
                r"(?i)datos[\s_]+de[\s_]+la[\s_]+vivienda",
                r"(?i)servicios[\s_]+(?:del[\s_]+)?hogar",
            ]
        else:
            key_words = [
                r"(?i)datos[\s_]+(?:de[\s_]+)?identificaci[oó]n",
                r"(?i)servicios[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)composici[oó]n[\s_]+(?:del[\s_]+)?hogar",
                r"(?i)datos[\s_]+(?:de[\s_]+)?(?:la[\s_]+)?vivienda",
            ]
        
        extractor = Extractor(
            input_path=url,
            down_ext=['.sav', '.zip'],
            sep=' ',
            output_path = f"data/ECV/ECV_{year}",
            depth=0,
            key_words=key_words,
            delete_zip_after=True
        )

        df_extracted = extractor.s4h_extract()
        
        dfs_vivienda_year = []
        dfs_hogar_year = []
        
        for df in df_extracted:
            df.columns = df.columns.str.strip()
            df.columns = [col.upper() for col in df.columns]
            df['YEAR'] = year
            
            # Normalizar nombres
            rename_map = {}
            if 'CANT_HOGARES_VIVIENDA' not in df.columns and 'CANT_HOG_COMPLETOS' in df.columns:
                rename_map['CANT_HOG_COMPLETOS'] = 'CANT_HOGARES_VIVIENDA'
            if rename_map:
                df = df.rename(columns=rename_map)
            
            # Renombrar columnas
            if 'DPTO' in df.columns and 'P1_DEPARTAMENTO' not in df.columns:
                df = df.rename(columns={'DPTO': 'P1_DEPARTAMENTO'})
            if 'P3' in df.columns and 'CLASE' not in df.columns:
                df = df.rename(columns={'P3': 'CLASE'})
            
            # Determinar si es tabla de vivienda o hogar
            has_vivienda_vars = any(col in df.columns for col in ['P4000', 'P4005', 'P4015', 'P8520'])
            has_hogar_vars = any(col in df.columns for col in ['P205', 'P5010', 'P8526', 'P8530', 'P8532', 'CANT_PERSONAS_HOGAR'])
            
            if 'DIRECTORIO' not in df.columns or df['DIRECTORIO'].isna().all():
                continue
            
            if has_vivienda_vars and not has_hogar_vars:
                dfs_vivienda_year.append(df)
            elif has_hogar_vars:
                dfs_hogar_year.append(df)
            elif has_vivienda_vars and has_hogar_vars:
                dfs_vivienda_year.append(df)
        
        # Merge de tablas de vivienda del mismo año
        if dfs_vivienda_year:
            if len(dfs_vivienda_year) == 1:
                merged_vivienda = dfs_vivienda_year[0]
            else:
                common_cols = set(dfs_vivienda_year[0].columns)
                for df in dfs_vivienda_year[1:]:
                    common_cols = common_cols.intersection(set(df.columns))
                common_cols.discard('DIRECTORIO')
                
                for i in range(1, len(dfs_vivienda_year)):
                    cols_to_drop = [col for col in common_cols if col in dfs_vivienda_year[i].columns]
                    if cols_to_drop:
                        dfs_vivienda_year[i] = dfs_vivienda_year[i].drop(columns=cols_to_drop)
                
                merged_vivienda = reduce(
                    lambda left, right: pd.merge(left, right, on='DIRECTORIO', how='outer'), 
                    dfs_vivienda_year
                )
            
            # Filtrar solo columnas de vivienda
            merged_vivienda = merged_vivienda[[col for col in vivienda_cols if col in merged_vivienda.columns]]
            dfs_vivienda_by_year[year] = merged_vivienda
            print(f"  -> Vivienda {year}: {len(merged_vivienda)} viviendas, columnas: {list(merged_vivienda.columns)}")
        
        # Merge de tablas de hogar del mismo año
        if dfs_hogar_year:
            if len(dfs_hogar_year) == 1:
                merged_hogar = dfs_hogar_year[0]
            else:
                common_cols = set(dfs_hogar_year[0].columns)
                for df in dfs_hogar_year[1:]:
                    common_cols = common_cols.intersection(set(df.columns))
                common_cols.discard('DIRECTORIO')
                
                for i in range(1, len(dfs_hogar_year)):
                    cols_to_drop = [col for col in common_cols if col in dfs_hogar_year[i].columns]
                    if cols_to_drop:
                        dfs_hogar_year[i] = dfs_hogar_year[i].drop(columns=cols_to_drop)
                
                merged_hogar = reduce(
                    lambda left, right: pd.merge(left, right, on='DIRECTORIO', how='outer'), 
                    dfs_hogar_year
                )
            
            # Filtrar solo columnas de hogar
            merged_hogar = merged_hogar[[col for col in hogar_cols if col in merged_hogar.columns]]
            dfs_hogar_by_year[year] = merged_hogar
            print(f"  -> Household {year}: {len(merged_hogar)} households, columns: {list(merged_hogar.columns)}")

2026-10-06 19:05:04,115 - INFO - ----------------------
2026-10-06 19:05:04,115 - INFO - Starting data extraction...
2026-10-06 19:05:04,115 - INFO - Extracting data in online mode...
2026-10-06 19:05:04,116 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0



2010: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata


2026-10-06 19:05:06,149 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata
2026-10-06 19:05:06,151 - INFO - Downloading files to: data/ECV/ECV_2010
2026-10-06 19:05:07,606 - INFO - Processing (depth 0): Caracteristicas y Composicion del Hogar.zip
2026-10-06 19:05:07,633 - INFO - Extracted: 45dc339a_Caracteristicas y Composición del Hogar_Dbfp_encv_547_1.sav
2026-10-06 19:05:07,635 - INFO - Processing (depth 0): Datos de Identificacion.zip
2026-10-06 19:05:07,646 - INFO - Extracted: bf00799c_Datos de Identificación_Dbfp_encv_545_1.sav
2026-10-06 19:05:07,647 - INFO - Processing (depth 0): Datos de la Vivienda.zip
2026-10-06 19:05:07,661 - INFO - Extracted: f8cc96cd_Datos de la Vivienda_Dbfp_encv_545_2.sav
2026-10-06 19:05:07,663 - INFO - Processing (depth 0): Servicios del Hogar.zip
2026-10-06 19:05:07,686 - INFO - Extracted: e735ac75_Servicios del Hogar_Dbfp_encv_546_1.sav
Processing files: 100%|███████████████████████████

  -> Vivienda 2010: 14268 viviendas, columnas: ['DIRECTORIO', 'YEAR', 'P4000', 'P4005', 'P4015', 'P8520S1', 'P8520S3', 'P8520S4', 'P8520S5', 'P70']
  -> Household 2010: 14801 households, columns: ['DIRECTORIO', 'YEAR', 'P205', 'P5010', 'P8526', 'P8530', 'P8532']

2011: https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata


2026-10-06 19:05:09,934 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata
2026-10-06 19:05:09,937 - INFO - Downloading files to: data/ECV/ECV_2011
2026-10-06 19:05:10,663 - INFO - Processing (depth 0): Datos de identificacion y de vivienda.zip
2026-10-06 19:05:10,682 - INFO - Extracted: a4f12337_Datos de identificación y de vivienda_Dbfp_encv_550_1.sav
2026-10-06 19:05:10,684 - INFO - Processing (depth 0): Datos de Servicios del hogar.zip
2026-10-06 19:05:10,717 - INFO - Extracted: f6ab06fb_Datos de Servicios del hogar_Dbfp_encv_551_1.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 11.04it/s]
2026-10-06 19:05:10,902 - INFO - Successfully processed 2/2 files
2026-10-06 19:05:10,902 - INFO - Deleted zip file after extraction: data/ECV/ECV_2011\Datos de identificacion y de vivienda.zip
2026-10-06 19:05:10,902

...

  -> Vivienda 2015: 22524 viviendas, columnas: ['DIRECTORIO', 'YEAR', 'CLASE', 'REGION', 'P1_DEPARTAMENTO', 'P4005', 'P4015', 'P8520S1', 'P8520S3', 'P8520S4', 'P8520S5', 'CANT_HOGARES_VIVIENDA']
  -> Household 2015: 23005 households, columns: ['DIRECTORIO', 'YEAR', 'CANT_PERSONAS_HOGAR', 'P5010', 'P8526', 'P8530', 'P764']

2016: https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata


2026-10-06 19:05:27,132 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata
2026-10-06 19:05:27,136 - INFO - Downloading files to: data/ECV/ECV_2016
2026-10-06 19:05:28,288 - INFO - Processing (depth 0): Datos de la vivienda.zip
2026-10-06 19:05:28,306 - INFO - Extracted: eb1546ff_Datos de la vivienda.sav
2026-10-06 19:05:28,309 - INFO - Processing (depth 0): Servicios del hogar.zip
2026-10-06 19:05:28,332 - INFO - Extracted: e2cc05de_Servicios del hogar.sav
2026-10-06 19:05:28,333 - INFO - Processing (depth 0): Caracteristicas y composicion del hogar.zip
2026-10-06 19:05:28,370 - INFO - Extracted: 2a0c6f3c_Características y composición del hogar.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00,  8.62it/s]
2026-10-06 19:05:28,721 - INFO - Successfully processed 3/3 files
2026-10-06 19:05:28,721 - INFO - Delete

  -> Vivienda 2016: 22454 viviendas, columnas: ['DIRECTORIO', 'YEAR', 'CLASE', 'REGION', 'P1_DEPARTAMENTO', 'P4005', 'P4015', 'P8520S1', 'P8520S3', 'P8520S4', 'P8520S5', 'CANT_HOGARES_VIVIENDA']
  -> Household 2016: 22893 households, columns: ['DIRECTORIO', 'YEAR', 'CANT_PERSONAS_HOGAR', 'P5010', 'P8526', 'P8530', 'P764']

2017: https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata


2026-10-06 19:05:30,704 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata
2026-10-06 19:05:30,706 - INFO - Downloading files to: data/ECV/ECV_2017
2026-10-06 19:05:31,709 - INFO - Processing (depth 0): Servicios del hogar Actualizada.zip
2026-10-06 19:05:31,728 - INFO - Extracted: ae52f600_Servicios del hogar Actualizada.sav
2026-10-06 19:05:31,731 - INFO - Processing (depth 0): Caracteristicas y composicion del hogar Actualizada.zip
2026-10-06 19:05:31,755 - INFO - Extracted: b7d33b2c_Caracteristicas y composicion del hogar Actualizada.sav
2026-10-06 19:05:31,757 - INFO - Processing (depth 0): Datos de la vivienda Actualizada.zip
2026-10-06 19:05:31,771 - INFO - Extracted: 11f2b128_Datos de la vivienda Actualizada.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 18.39it/s]
2026-10-06 19:05:31,937 - INFO - 

  -> Vivienda 2017: 8501 viviendas, columnas: ['DIRECTORIO', 'YEAR', 'CLASE', 'P4005', 'P4015', 'P8520S1', 'P8520S3', 'P8520S4', 'P8520S5', 'CANT_HOGARES_VIVIENDA']
  -> Household 2017: 8612 households, columns: ['DIRECTORIO', 'YEAR', 'CANT_PERSONAS_HOGAR', 'P5010', 'P8526', 'P8530', 'P764']


## 4. Extraction of Expansion Factors

In [7]:
dfs_fex_by_year = {}
for year, url in ecv_data.items():
    if 2010 <= year <= 2018:
        print(f"\n{year}: Extracting of Expansion Factors...")
        key_words_fex_2018 = [
            r"(?i)factores[\s_]+de[\s_]+expansi[oó]n[\s_]+(?:20)?[0-9]{2}[\s_]+basados[\s_]+en[\s_]+el[\s_]+CNPV[\s_]+2018",
        ]

        extractor_fex_2018 = Extractor(
            input_path=url,
            down_ext=['.sav', '.zip'],
            sep=' ',
            output_path = f"data/ECV/ECV_FEX_2018_{year}",
            depth=0,
            key_words=key_words_fex_2018,
            delete_zip_after=True
        )

        try:
            df_extracted_fex_2018 = extractor_fex_2018.s4h_extract()
            if df_extracted_fex_2018:
                df_fex = df_extracted_fex_2018[0]
                df_fex.columns = df_fex.columns.str.strip()
                df_fex.columns = [col.upper() for col in df_fex.columns]
                
                if 'DIRECTORIO' in df_fex.columns and 'FEX_C_2018' in df_fex.columns:
                    df_fex = df_fex[['DIRECTORIO', 'FEX_C_2018']].drop_duplicates(subset=['DIRECTORIO'])
                    dfs_fex_by_year[year] = df_fex
                    print(f"  -> FEX {year}: {len(df_fex)} registers")
        except Exception as e:
            print(f"Skipping year {year} for FEX_C_2018 extraction: {e}")

2026-10-06 19:05:52,447 - INFO - ----------------------
2026-10-06 19:05:52,448 - INFO - Starting data extraction...
2026-10-06 19:05:52,448 - INFO - Extracting data in online mode...
2026-10-06 19:05:52,448 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0



2010: Extrayendo factores de expansion...


2026-10-06 19:05:54,425 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata
2026-10-06 19:05:54,427 - INFO - Downloading files to: data/ECV/ECV_FEX_2018_2010
2026-10-06 19:05:54,717 - INFO - Processing (depth 0): Factores de expansion 2010 basados en el CNPV 2018.zip
2026-10-06 19:05:54,730 - INFO - Extracted: 04ae90c2_FACTORES_ECV_2010_CNPV2018.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 39.94it/s]
2026-10-06 19:05:54,758 - INFO - Successfully processed 1/1 files
2026-10-06 19:05:54,759 - INFO - Deleted zip file after extraction: data/ECV/ECV_FEX_2018_2010\Factores de expansion 2010 basados en el CNPV 2018.zip
2026-10-06 19:05:54,759 - INFO - Extraction completed successfully.
2026-10-06 19:05:54,763 - INFO - ----------------------
2026-10-06 19:05:54,763 - INFO - Starting data extraction...
2026-10-06 

  -> FEX 2010: 14268 registers

2011: Extrayendo factores de expansion...


2026-10-06 19:05:56,673 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata
2026-10-06 19:05:56,676 - INFO - Downloading files to: data/ECV/ECV_FEX_2018_2011
2026-10-06 19:06:12,077 - INFO - Processing (depth 0): Factores de expansion 2011 basados en el CNPV 2018.zip
2026-10-06 19:06:12,092 - INFO - Extracted: 6bcca20b_FACTORES_ECV_2011_CNPV2018.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.49it/s]
2026-10-06 19:06:12,141 - INFO - Successfully processed 1/1 files
2026-10-06 19:06:12,141 - INFO - Deleted zip file after extraction: data/ECV/ECV_FEX_2018_2011\Factores de expansion 2011 basados en el CNPV 2018.zip
2026-10-06 19:06:12,142 - INFO - Extraction completed successfully.
2026-10-06 19:06:12,144 - INFO - ----------------------
2026-10-06 19:06:12,146 - INFO - Starting data extraction...
2026-10-06 

...

  -> FEX 2015: 22524 registers

2016: Extrayendo factores de expansion...


2026-10-06 19:06:30,494 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata
2026-10-06 19:06:30,497 - INFO - Downloading files to: data/ECV/ECV_FEX_2018_2016
2026-10-06 19:06:30,815 - INFO - Processing (depth 0): Factores de expansion 2016 basados en el CNPV 2018.zip
2026-10-06 19:06:30,830 - INFO - Extracted: 4dd14e7d_FACTORES_ECV_2016_CNPV2018.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 27.76it/s]
2026-10-06 19:06:30,868 - INFO - Successfully processed 1/1 files
2026-10-06 19:06:30,869 - INFO - Deleted zip file after extraction: data/ECV/ECV_FEX_2018_2016\Factores de expansion 2016 basados en el CNPV 2018.zip
2026-10-06 19:06:30,869 - INFO - Extraction completed successfully.
2026-10-06 19:06:30,871 - INFO - ----------------------
2026-10-06 19:06:30,872 - INFO - Starting data extraction...
2026-10-06 

  -> FEX 2016: 22454 registers

2017: Extrayendo factores de expansion...


2026-10-06 19:06:32,781 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata
2026-10-06 19:06:32,783 - INFO - Downloading files to: data/ECV/ECV_FEX_2018_2017
2026-10-06 19:06:33,063 - INFO - Processing (depth 0): Factores de expansion 2017 basados en el CNPV 2018.zip
2026-10-06 19:06:33,073 - INFO - Extracted: 081be155_FACTORES_ECV_2017_CNPV2018.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 58.77it/s]
2026-10-06 19:06:33,092 - INFO - Successfully processed 1/1 files
2026-10-06 19:06:33,092 - INFO - Deleted zip file after extraction: data/ECV/ECV_FEX_2018_2017\Factores de expansion 2017 basados en el CNPV 2018.zip
2026-10-06 19:06:33,093 - INFO - Extraction completed successfully.


  -> FEX 2017: 8501 registers


## 5. List of Municipalities

In [9]:
dfs_mpio_by_year = {}
for year, url in ecv_data.items():
    print(f"\n{year}: Extrancting Municipalities...")
    extractor_mpio = Extractor(
        input_path=url,
        down_ext=['.sav', '.zip'],
        sep=' ',
        output_path = f"data/ECV/ECV_MPIO_{year}",
        depth=0,
        key_words=[
            r"(?i)municipio[\s_]+de[\s_]+aplicaci[oó]n[\s_]+de[\s_]+la[\s_]+encuesta[\s_]+(?:20)?[0-9]{2}",
        ],
        delete_zip_after=True
    )

    try:
        df_extracted_mpio = extractor_mpio.s4h_extract()
        if df_extracted_mpio:
            df_mpio = df_extracted_mpio[0]
            df_mpio.columns = df_mpio.columns.str.strip()
            df_mpio.columns = [col.upper() for col in df_mpio.columns]
            
            if 'DIRECTORIO' in df_mpio.columns and 'P1_MUNICIPIO' in df_mpio.columns:
                df_mpio = df_mpio[['DIRECTORIO', 'P1_MUNICIPIO']].drop_duplicates(subset=['DIRECTORIO'])
                dfs_mpio_by_year[year] = df_mpio
                print(f"  -> Municipalities {year}: {len(df_mpio)} registers")
    except Exception as e:
        print(f"Skipping year {year} for municipio extraction: {e}")

2026-10-06 19:08:02,357 - INFO - ----------------------
2026-10-06 19:08:02,357 - INFO - Starting data extraction...
2026-10-06 19:08:02,357 - INFO - Extracting data in online mode...
2026-10-06 19:08:02,358 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0



2010: Extrancting Municipalities...


2026-10-06 19:08:04,435 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata
2026-10-06 19:08:04,437 - WARNING - No downloadable files found matching criteria. Returning empty extraction.
2026-10-06 19:08:04,437 - INFO - Extraction completed successfully.
2026-10-06 19:08:04,437 - WARNING - No data was extracted. The extraction process returned an empty result.
2026-10-06 19:08:04,438 - INFO - ----------------------
2026-10-06 19:08:04,438 - INFO - Starting data extraction...
2026-10-06 19:08:04,439 - INFO - Extracting data in online mode...
2026-10-06 19:08:04,439 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata with depth 0



2011: Extrancting Municipalities...


2026-10-06 19:08:06,401 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/196/get-microdata
2026-10-06 19:08:06,402 - WARNING - No downloadable files found matching criteria. Returning empty extraction.
2026-10-06 19:08:06,402 - INFO - Extraction completed successfully.
2026-10-06 19:08:06,402 - WARNING - No data was extracted. The extraction process returned an empty result.
2026-10-06 19:08:06,404 - INFO - ----------------------
2026-10-06 19:08:06,404 - INFO - Starting data extraction...
2026-10-06 19:08:06,404 - INFO - Extracting data in online mode...
2026-10-06 19:08:06,404 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/124/get-microdata with depth 0


...

2026-10-06 19:08:14,160 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/419/get-microdata
2026-10-06 19:08:14,162 - WARNING - No downloadable files found matching criteria. Returning empty extraction.
2026-10-06 19:08:14,162 - INFO - Extraction completed successfully.
2026-10-06 19:08:14,162 - WARNING - No data was extracted. The extraction process returned an empty result.
2026-10-06 19:08:14,163 - INFO - ----------------------
2026-10-06 19:08:14,164 - INFO - Starting data extraction...
2026-10-06 19:08:14,164 - INFO - Extracting data in online mode...
2026-10-06 19:08:14,164 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata with depth 0



2016: Extrancting Municipalities...


2026-10-06 19:08:16,151 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/456/get-microdata
2026-10-06 19:08:16,152 - WARNING - No downloadable files found matching criteria. Returning empty extraction.
2026-10-06 19:08:16,153 - INFO - Extraction completed successfully.
2026-10-06 19:08:16,153 - WARNING - No data was extracted. The extraction process returned an empty result.
2026-10-06 19:08:16,154 - INFO - ----------------------
2026-10-06 19:08:16,154 - INFO - Starting data extraction...
2026-10-06 19:08:16,155 - INFO - Extracting data in online mode...
2026-10-06 19:08:16,155 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata with depth 0



2017: Extrancting Municipalities...


2026-10-06 19:08:18,146 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/544/get-microdata
2026-10-06 19:08:18,147 - WARNING - No downloadable files found matching criteria. Returning empty extraction.
2026-10-06 19:08:18,147 - INFO - Extraction completed successfully.
2026-10-06 19:08:18,147 - WARNING - No data was extracted. The extraction process returned an empty result.


## 6. 2010 Identity Extraction

In [10]:
dfs_2010_id = []
for year, url in ecv_data.items():
    if 2010 == year:
        print(f"\n{year}: Extracting <<identificacion-final>>...")
        key_words_2010_id = ["Datos de idenrificacion-final"]
        extractor_2010_id = Extractor(
            input_path=url,
            down_ext=['.sav', '.zip'],
            sep=' ',
            output_path = f"data/ECV/ECV_2010_ID_{year}",
            depth=0,
            key_words=key_words_2010_id,
            delete_zip_after=True
        )

        try:
            dfs_2010_id = extractor_2010_id.s4h_extract()
            for df in dfs_2010_id:
                df.columns = df.columns.str.strip()
                df.columns = [col.upper() for col in df.columns]
        except Exception as e:
            print(f"Skipping year {year} for 2010_ID extraction: {e}")

2026-10-06 19:09:05,268 - INFO - ----------------------
2026-10-06 19:09:05,269 - INFO - Starting data extraction...
2026-10-06 19:09:05,269 - INFO - Extracting data in online mode...
2026-10-06 19:09:05,270 - INFO - Scraping URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata with depth 0



2010: Extracting <<identificacion-final>>...


2026-10-06 19:09:07,269 - INFO - Spider completed successfully for URL: https://microdatos.dane.gov.co/index.php/catalog/201/get-microdata
2026-10-06 19:09:07,272 - INFO - Downloading files to: data/ECV/ECV_2010_ID_2010
2026-10-06 19:09:07,564 - INFO - Processing (depth 0): Datos de idenrificacion-final.zip
2026-10-06 19:09:07,579 - INFO - Extracted: 5fcda8a9_Datos de idenrificación-final_Dbfp_encv_545_1.sav
Processing files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 57.12it/s]
2026-10-06 19:09:07,599 - INFO - Successfully processed 1/1 files
2026-10-06 19:09:07,599 - INFO - Deleted zip file after extraction: data/ECV/ECV_2010_ID_2010\Datos de idenrificacion-final.zip
2026-10-06 19:09:07,600 - INFO - Extraction completed successfully.


## 7. Processing and Consolidation by Year

In [12]:
for year in ecv_data.keys():
        print(f"\nProcessing {year}...")
        
        # Merge with expansion factor (only in df_hogar)
        if year in dfs_hogar_by_year and year in dfs_fex_by_year:
            df_hogar = dfs_hogar_by_year[year]
            df_fex = dfs_fex_by_year[year]
            
            df_hogar = df_hogar.merge(df_fex, on='DIRECTORIO', how='left')
            dfs_hogar_by_year[year] = df_hogar
            print(f"  -> Merge FEX_C_2018 finished in df_hogar")
        
        # Merge with municipality
        if year in dfs_vivienda_by_year and year in dfs_mpio_by_year:
            df_vivienda = dfs_vivienda_by_year[year]
            df_mpio = dfs_mpio_by_year[year]
            
            if 'P1_MUNICIPIO' not in df_vivienda.columns or df_vivienda['P1_MUNICIPIO'].isna().all():
                df_vivienda = df_vivienda.merge(df_mpio, on='DIRECTORIO', how='left')
                dfs_vivienda_by_year[year] = df_vivienda
                print(f"  -> Merge P1_MUNICIPIO finished")
        
        # Merge with identification 2010
        if year == 2010 and dfs_2010_id:
            df_2010_id = dfs_2010_id[0]
            
            if 'P3' in df_2010_id.columns and 'CLASE' not in df_2010_id.columns:
                df_2010_id = df_2010_id.rename(columns={'P3': 'CLASE'})
            
            rename_map = {}
            if 'P1_DEPARTAMENTO' not in df_2010_id.columns and 'DEPARTAMENTO' in df_2010_id.columns:
                rename_map['DEPARTAMENTO'] = 'P1_DEPARTAMENTO'
            if rename_map:
                df_2010_id = df_2010_id.rename(columns=rename_map)
            
            if 'DIRECTORIO' in df_2010_id.columns:
                df_2010_id = df_2010_id.drop_duplicates(subset=['DIRECTORIO'], keep='first')
                
                cols_to_merge = ['DIRECTORIO']
                for col in ['P1_DEPARTAMENTO', 'CLASE', 'REGION']:
                    if col in df_2010_id.columns:
                        cols_to_merge.append(col)
                
                if len(cols_to_merge) > 1 and year in dfs_vivienda_by_year:
                    df_vivienda = dfs_vivienda_by_year[year]
                    
                    df_vivienda = df_vivienda.merge(
                        df_2010_id[cols_to_merge], 
                        on='DIRECTORIO', 
                        how='left', 
                        suffixes=('', '_2010')
                    )
                    
                    for col in ['P1_DEPARTAMENTO', 'CLASE', 'REGION']:
                        col_2010 = f'{col}_2010'
                        if col_2010 in df_vivienda.columns:
                            if col in df_vivienda.columns:
                                df_vivienda[col] = df_vivienda[col].combine_first(df_vivienda[col_2010])
                            else:
                                df_vivienda[col] = df_vivienda[col_2010]
                            df_vivienda = df_vivienda.drop(columns=[col_2010])
                    
                    dfs_vivienda_by_year[year] = df_vivienda
                    print(f"  -> Merge identificacion 2010 finished")


Processing 2010...
  -> Merge FEX_C_2018 finished in df_hogar
  -> Merge identificacion 2010 finished

Processing 2011...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2012...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2013...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2014...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2015...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2016...
  -> Merge FEX_C_2018 finished in df_hogar

Processing 2017...
  -> Merge FEX_C_2018 finished in df_hogar


## 8. Saving CSV Files

In [14]:
run_dir = Path("data/dfs") / f"run_{uuid.uuid4().hex[:8]}"
run_dir.mkdir(parents=True, exist_ok=False)

def convert_dtypes(df):
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        non_null = converted.dropna()

        if len(non_null) > 0 and (non_null % 1 == 0).all():
            df[col] = converted.astype("Int64")
        else:
            df[col] = converted

    return df

years = sorted(set(dfs_vivienda_by_year.keys()) | set(dfs_hogar_by_year.keys()))
for year in years:
    df_vivienda = dfs_vivienda_by_year.get(year)
    df_hogar = dfs_hogar_by_year.get(year)

    if df_vivienda is not None and df_hogar is not None:
        df_joined = df_vivienda.merge(
            df_hogar,
            on='DIRECTORIO',
            how='outer',
            suffixes=('_vivienda', '_hogar')
        )

        if 'YEAR_vivienda' in df_joined.columns:
            df_joined = df_joined.rename(columns={'YEAR_vivienda': 'YEAR'})
        if 'YEAR_hogar' in df_joined.columns:
            if 'YEAR' in df_joined.columns:
                df_joined['YEAR'] = df_joined['YEAR'].combine_first(df_joined['YEAR_hogar'])
            else:
                df_joined = df_joined.rename(columns={'YEAR_hogar': 'YEAR'})
            df_joined = df_joined.drop(columns=['YEAR_hogar'])
    elif df_vivienda is not None:
        df_joined = df_vivienda.copy()
    elif df_hogar is not None:
        df_joined = df_hogar.copy()
    else:
        continue

    df_joined = convert_dtypes(df_joined)

    output_file = run_dir / f"df_ecv_{year}.csv"
    df_joined.to_csv(output_file, index=False)
    print(f"  -> {year}: {len(df_joined)} filas guardadas en {output_file.name}")

print(f"\nProcess completed. Files available at: {run_dir}")


if extractor is not None:
    extractor.s4h_delete_download_folder(folder_path="data/ECV")

  -> 2010: 14801 filas guardadas en df_ecv_2010.csv
  -> 2011: 25364 filas guardadas en df_ecv_2011.csv
  -> 2012: 21383 filas guardadas en df_ecv_2012.csv
  -> 2013: 21565 filas guardadas en df_ecv_2013.csv
  -> 2014: 20141 filas guardadas en df_ecv_2014.csv
  -> 2015: 23005 filas guardadas en df_ecv_2015.csv


2026-10-06 19:13:34,977 - INFO - Deleting folder: data\ECV
2026-10-06 19:13:34,996 - INFO - Folder deleted successfully


  -> 2016: 22893 filas guardadas en df_ecv_2016.csv
  -> 2017: 8612 filas guardadas en df_ecv_2017.csv

Process completed. Files available at: data\dfs\run_78ce89c5
